# `q_diagonalize()`

`nematics3d.q_diagonalize()` recovers named scalar-order and director results from $Q$-tensor data and can optionally return the complete biaxial eigensystem.

## What `q_diagonalize()` is for

A three-dimensional nematic configuration may be stored as a $Q$-tensor field rather than as separate scalar-order and director fields. `q_diagonalize()` performs the conversion in the other direction. Its result object always contains the scalar order parameter $S$, the dominant director $\mathbf{n}$, and indices that identify specially handled points.

Use this function when $Q$ is the available representation but a later calculation, inspection, or visualization requires $S$ and $\mathbf{n}$. Set `is_biaxial=True` when the complete eigenvalues, eigenvectors, and biaxial order are also needed.

The returned director represents a nematic axis, so $\mathbf{n}$ and $-\mathbf{n}$ describe the same physical state. At an isotropic point, where $Q$ does not define a preferred axis, the returned director is only a deterministic numerical placeholder.

## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** This tutorial constructs small $Q$-tensor examples directly, so no external data files or visualization setup are required. The code cell below imports NumPy for constructing and inspecting arrays and imports `nematics3d` through its public package interface.

In [ ]:
import numpy as np

import nematics3d

## Minimal example

We start with one full $3 \times 3$ $Q$ tensor whose dominant axis is parallel to $(1,1,1)$ and whose scalar order is $S=1$. The function returns a `QDiagonalizationResult`, so outputs are read by name.

In [ ]:
Q = np.array(
    [
        [0.0, 1.0 / 3.0, 1.0 / 3.0],
        [1.0 / 3.0, 0.0, 1.0 / 3.0],
        [1.0 / 3.0, 1.0 / 3.0, 0.0],
    ]
)

result = nematics3d.q_diagonalize(Q)
S = result.S
n = result.n

print("S =", S)
print("n =", n)

The recovered value is $S = 1$, and the director is parallel to $(1, 1, 1)$. The displayed components may all have the opposite sign from the direction used to construct the tensor. This does not change the physical result because a nematic director satisfies $\mathbf{n} \equiv -\mathbf{n}$.

A single tensor produces one scalar and one three-component director. Later sections will extend the same call to arrays of $Q$ tensors and explain the accepted compact and full tensor representations.

### What is returned

`q_diagonalize()` returns a `QDiagonalizationResult`. Its fields are:

- `S`: scalar order parameter $S$;
- `n`: dominant director $\mathbf{n}$;
- `isotropic_indices`: index tuples classified as numerically isotropic;
- `uniaxial_indices`: index tuples receiving canonical uniaxial-frame treatment when `is_biaxial=True`;
- `eigenvalues`, `eigenvectors`, and `biaxial_order`: `None` by default and populated when `is_biaxial=True`.

Named fields avoid relying on tuple position. The director is normalized, but its overall sign is not fixed.

## Arguments

The public signature is:

```python
q_diagonalize(qtensor, *, is_biaxial=False)
```

### `qtensor`

`qtensor` must be a non-empty floating-point array with trailing shape `(..., 5)` or `(..., 3, 3)`. The compact component order is $(Q_{xx},Q_{xy},Q_{xz},Q_{yy},Q_{yz})$. Leading dimensions are preserved in the returned fields.

Full tensors are validated as finite, symmetric, and traceless. Compact tensors are symmetric and traceless by construction and are checked for finite values.

### `is_biaxial`

The default `False` uses the faster path that returns only the dominant physical quantities and diagnostic indices. Set it to `True` to obtain all three descending eigenvalues, their matching eigenvector columns, and the biaxial order.

## Examples

### Example: use the compact five-component representation

A symmetric traceless $3 \times 3$ tensor has five independent components. The compact representation stores them in the fixed order

$$
(Q_{xx}, Q_{xy}, Q_{xz}, Q_{yy}, Q_{yz}).
$$

The array `Q_compact` below represents the same mathematical tensor $Q$ used in the minimal example. The omitted entries are reconstructed from symmetry, and $Q_{zz}$ is reconstructed from the traceless condition.

In [ ]:
Q_compact = np.array(
    [
        0.0,  # Q_xx
        1.0 / 3.0,  # Q_xy
        1.0 / 3.0,  # Q_xz
        0.0,  # Q_yy
        1.0 / 3.0,  # Q_yz
    ]
)

compact_result = nematics3d.q_diagonalize(Q_compact)
same_order = np.isclose(compact_result.S, result.S)
same_axis = np.isclose(abs(np.dot(compact_result.n, result.n)), 1.0)

print("S =", compact_result.S)
print("n =", compact_result.n)
print("same order parameter =", same_order)
print("same director axis =", same_axis)

Both representations recover the same scalar order and the same nematic axis. The axis comparison remains sign-independent because the compact and full calculations are each free to return either representative of $\mathbf{n} \equiv -\mathbf{n}$.

The compact form is useful for storing a symmetric traceless $Q$ field without carrying four redundant matrix entries. Use the full representation when an explicit matrix is more convenient or is already produced by the preceding calculation.

### Example: diagonalize a lattice of tensors at once

The most common input is a $Q$ field sampled on an $(N_x, N_y, N_z, \ldots)$ lattice. With the full representation, such an array typically has shape `(Nx, Ny, Nz, ..., 3, 3)`; with the compact representation, it has shape `(Nx, Ny, Nz, ..., 5)`. `q_diagonalize()` operates on all lattice dimensions at once, so there is no need to write a Python loop over individual points.

The following two-point array is the smallest illustration of this rule. It contains the original tensor $Q$ and a second tensor with half its magnitude. Scaling $Q$ by $1/2$ keeps the same nematic axis while reducing $S$ by the same factor.

In [ ]:
Q_batch = np.stack([Q, 0.5 * Q])
batch_result = nematics3d.q_diagonalize(Q_batch)

print("Q_batch shape =", Q_batch.shape)
print("S shape =", batch_result.S.shape)
print("S =", batch_result.S)
print("n shape =", batch_result.n.shape)
print("n =")
print(batch_result.n)

Here the input shape `(2, 3, 3)` consists of a leading batch dimension of length two followed by the two matrix axes. `q_diagonalize()` preserves that leading dimension, so $S$ has shape `(2,)` and $\mathbf{n}$ has shape `(2, 3)`.

The result $S=[1, 0.5]$ shows the expected linear change in scalar order. Both rows of $\mathbf{n}$ describe the same axis. Their signs happen to agree here, but downstream comparisons should still treat $\mathbf{n}$ and $-\mathbf{n}$ as equivalent. For a lattice with shape `(Nx, Ny, Nz, ...)`, the output $S$ has shape `(Nx, Ny, Nz, ...)`, while $\mathbf{n}$ has shape `(Nx, Ny, Nz, ..., 3)`.

### Example: request biaxial information

For a uniaxial tensor, $S$ and $\mathbf{n}$ are sufficient to reconstruct $Q$. A biaxial tensor has two different eigenvalues perpendicular to the principal axis. Pass `is_biaxial=True` to preserve that additional information.

The example below has eigenvalues $0.6$, $-0.1$, and $-0.5$, with axes rotated away from the coordinate axes.

In [ ]:
principal_axis = np.array([1.0, 1.0, 1.0]) / np.sqrt(3.0)
second_axis = np.array([1.0, -1.0, 0.0]) / np.sqrt(2.0)
third_axis = np.cross(principal_axis, second_axis)
biaxial_axes = np.column_stack([principal_axis, second_axis, third_axis])
Q_biaxial = biaxial_axes @ np.diag([0.6, -0.1, -0.5]) @ biaxial_axes.T

biaxial_result = nematics3d.q_diagonalize(Q_biaxial, is_biaxial=True)

print("S =", biaxial_result.S)
print("n =", biaxial_result.n)
print("eigenvalues =", biaxial_result.eigenvalues)
print("biaxial order =", biaxial_result.biaxial_order)

The eigenvalues are returned in descending order, and the columns of `eigenvectors` match that order. Here the largest eigenvalue is $0.6$, so $S=0.9$. The biaxial order is

$$
b=\frac{3}{2}|\lambda_1-\lambda_2|,
$$

where $\lambda_1$ and $\lambda_2$ are the two lower eigenvalues. Thus this example gives $b=0.6$.

## Special cases

### Example: handle an isotropic point

At an isotropic point, $Q=0$ and $S=0$, so no unique physical director exists. `q_diagonalize()` returns a deterministic placeholder and records the location in `isotropic_indices`.

In [ ]:
Q_isotropic = np.zeros((3, 3), dtype=float)
isotropic_result = nematics3d.q_diagonalize(Q_isotropic)

print("S =", isotropic_result.S)
print("n =", isotropic_result.n)
print("isotropic indices =", isotropic_result.isotropic_indices)

For a single tensor, `[()]` means that its zero-dimensional position is isotropic, while `[]` means that no position is isotropic. For a field, each tuple contains the full lattice coordinate of one isotropic point, such as `(i, j, k)`. Use these indices directly to inspect locations where $\mathbf{n}$ is not physically defined.

### Example: handle a purely uniaxial tensor

A purely uniaxial tensor has one unique principal axis and a degenerate two-dimensional perpendicular eigenspace. When `is_biaxial=True`, `q_diagonalize()` sets the two repeated lower eigenvalues exactly to half the negative largest eigenvalue, sets the biaxial order to zero, and constructs a deterministic right-handed orthonormal complement.

The perpendicular vectors are not unique physical directions and may be discontinuous between neighboring points. The affected points are listed in `uniaxial_indices`. This example uses a one-point batch so its point index is shown explicitly as `(0,)`.

In [ ]:
S_uniaxial = 0.75
n_uniaxial = np.array([1.0, 1.0, 1.0]) / np.sqrt(3.0)
Q_uniaxial = S_uniaxial * (
    np.outer(n_uniaxial, n_uniaxial) - np.eye(3) / 3.0
)

uniaxial_result = nematics3d.q_diagonalize(
    Q_uniaxial[None, ...],
    is_biaxial=True,
)

print("eigenvalues =", uniaxial_result.eigenvalues[0])
print("biaxial order =", uniaxial_result.biaxial_order[0])
print("uniaxial indices =", uniaxial_result.uniaxial_indices)

### Example: handle a principal director aligned with the $x$ axis

Alignment with the $x$ axis is a separate numerical special case: the cofactor expression used by the fast director-only path becomes zero even when the tensor is biaxial and its principal eigenvector is unique. `q_diagonalize()` detects the unstable analytic vector and recomputes only that point with `np.linalg.eigh()`.

The tensor below is deliberately biaxial, with three distinct eigenvalues, so this example isolates $x$-axis alignment from uniaxial degeneracy.

In [ ]:
Q_x = np.diag([0.6, -0.1, -0.5])
x_result = nematics3d.q_diagonalize(Q_x)

print("S =", x_result.S)
print("n =", x_result.n)
print("uniaxial indices =", x_result.uniaxial_indices)

## Details

**Readers who only need to use `q_diagonalize()` can safely skip this section.** This section records implementation relationships useful to maintainers.

### Call relationships inside Nematics3D

`q_diagonalize()` directly calls:

- `as_qfield9()`

The following functions and methods directly call `q_diagonalize()`:

- `QFieldObject.__init__()`
- `QPlane.act_refresh()`
- `QPlane.act_calc_omega()`
- `QSurface.act_refresh()`
- `defect_detect_surface()`
- `nml_principal_plane_analysis()`

### Implementation outline

The default path computes the tensor invariants and dominant eigenpair with vectorized NumExpr expressions, avoiding a full-field call to `np.linalg.eigh()`. Numerically exceptional points use a local eigensolver fallback.

When `is_biaxial=True`, all three analytic roots are sorted in descending order. Stable eigenvectors are constructed from the null spaces of $Q-\lambda I$; only exceptional points use the fallback. Positive-$S$ uniaxial points receive an exact repeated lower eigenvalue pair and a deterministic orthonormal complement.

## Possible issues

### Negative-$S$ oblate uniaxial states are not supported

The uniaxial special treatment assumes a unique largest eigenvalue and a repeated lower pair, as in a positive-$S$ prolate state.

### The sign of the director may change

The vectors $\mathbf{n}$ and $-\mathbf{n}$ represent the same nematic axis. Compare axes with a sign-independent measure.

### Isotropic directors are placeholders

Use `isotropic_indices` to identify points where the returned director has no physical orientation.

### Degenerate perpendicular axes may be discontinuous

At points in `uniaxial_indices`, the two perpendicular eigenvectors are a deterministic orthonormal complement, not unique physical directions. They can change discontinuously between neighboring points.

### Complete biaxial output costs more

Keep the default `is_biaxial=False` when only $S$ and $\mathbf{n}$ are needed. Requesting the complete eigensystem allocates and computes additional arrays.

## Useful links

- [`getQ()`](../../src/nematics3d/field.py) constructs a uniaxial $Q$ field from $S$ and $\mathbf{n}$.
- [`Q_diagonalize_linalg()`](../../src/nematics3d/field.py) is the lower-level NumPy eigensystem helper.
- [`QFieldObject`](../classes/QFieldObject/QFieldObject.ipynb) provides the higher-level object interface for working with $Q$ fields.